In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
from torch.amp import autocast, GradScaler
from datetime import datetime

In [ ]:
# Seed torch for reproducibility
import os, random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load Data
# **IMPORTANT** Make sure ALL data files are either in the "Data" directory, or if stored locally, change path to local path
# spectra_data = np.load("G:\\Comp 562 Exoplanet Data Raw per 10000\\npy_outputs\\ALL_planet_spectra_combined.npy")
spectra_data = np.load("..\\Data\\spectra_data_first_30000.npy")

wavelengths = pd.read_csv("..\\Data\\wavelengths.csv", header=0).values.flatten().astype(np.float64)

#metadata = pd.read_csv("..\\PSG_models_all.csv")
metadata = pd.read_csv("..\\Data\\PSG_models_first_30000.csv")

In [ ]:
# Filter wavelength to < 2.0 microns and extract signal
wavelength_mask = wavelengths <= 2.0
filtered_signals = spectra_data[:, wavelength_mask]

In [ ]:
# Normalize spectra data
scaler = StandardScaler()
X = scaler.fit_transform(filtered_signals)

In [ ]:
# The model will run on all gas labels contained in this list one-at-a-time.
# If only wanting to run once for a singular gas, only include that gas in the list
# All 12 gas abundances are labeled in the dataset as follows: ['H2O', 'CO2', 'O2', 'N2', 'CH4', 'N2O', 'CO', 'O3', 'SO2', 'NH3', 'C2H6', 'NO2']
gas_labels = ['H2O', 'CO2', 'O2', 'N2', 'CH4', 'N2O', 'CO', 'O3', 'SO2', 'NH3', 'C2H6', 'NO2']

In [ ]:
# **IMPORTANT** This model is simplified to only plot predictions and compute metrics. If wanting more detailed output, please refer to the individual CNN.
# When a predictions vs actual plot is generated, it will not be shown here, and instead will be added to the "gas_plots" directory.
# Results will be saved as a .csv containing a table with the MSE and R^2 values for each gas.

results = []
os.makedirs("gas_plots", exist_ok=True)

In [ ]:
# Model Training and Validation loop

for gas in gas_labels:
    start = datetime.now()
    print(f"\n=== Running Model for {gas} ===")

    scaler = StandardScaler()
    X = scaler.fit_transform(filtered_signals)

    gas_data = metadata[gas].values

    # Filter out all missing data from the dataset
    mask = (~np.isnan(X).any(axis=1)) & (~np.isnan(gas_data))
    X_clean = X[mask]
    gas_clean = gas_data[mask]

    # Remove any abundances < 0.0 or >= 1.0 (abundances are fractional, adding up to 1, so any entry outside of these bounds should not be considered,)
    clip_mask = (gas_clean >= 0.0) & (gas_clean < 1.0)
    X = X_clean[clip_mask]
    Y = np.log10(gas_clean[clip_mask] + 1e-8)

    # Train/val/test split
    # If running on the full dataset of 300k, splits should be:
    #   - Training set: 272,700     ~ 90.9% of the dataset
    #   - Validation set: 24,800    ~ 8.27% 
    #   - Testing set: 2,500        ~ 0.83%
    # If not, still use same percentage based splits
    X_train, X_temp, Y_train, Y_temp = train_test_split(X, Y, train_size=(272700 / 300000), random_state=42)
    X_val, X_test, Y_val, Y_test = train_test_split(X_temp, Y_temp, train_size=(24800 / 27300), test_size=(2500 / 27300), random_state=42)

    # Dataset
    class SpectraDataset(torch.utils.data.Dataset):
        def __init__(self, X, Y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.Y = torch.tensor(Y, dtype=torch.float32).unsqueeze(1)

        def __len__(self):
            return len(self.Y)

        def __getitem__(self, idx):
            return self.X[idx], self.Y[idx]
    
    # CNN
    # Architecture: Conv1d(64)-tanh-MaxPool-Conv1d(64)-relu-MaxPool-Conv1d(128)-reluMaxPool-Conv1d(256)-relu-FC(256)-relu-FC(12)
    class SpectraNet(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.cnn = nn.Sequential(
                nn.Conv1d(1, 64, kernel_size=3, padding=1),
                nn.Tanh(),
                nn.MaxPool1d(2),

                nn.Conv1d(64, 64, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool1d(2),

                nn.Conv1d(64, 128, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.MaxPool1d(2),

                nn.Conv1d(128, 256, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1)
            )
            self.fc = nn.Sequential(
                nn.Flatten(),
                nn.Linear(256, 256),
                nn.ReLU(),
                nn.Dropout(0.1),
                nn.Linear(256, 1)
            )

        def forward(self, x):
            x = x.unsqueeze(1)
            x = self.cnn(x)
            return self.fc(x)

    # Model setup and hyperparameter preparation
    # Current parameters:
    #   - Optimizer: Adam
    #   - Loss: MSE
    #   - Learning-rate: 6.58e-4 (Found via torch's LRFinder) with lr_scheduler utilization
    #   - Batch-size: 64 (Can go up to 128 to save some time, but not recommended for bet results)
    #   - Epochs: 150 (Realistically could go down to 100, but found best results at 150 using early stopping)
    #   - Uses GradScaler() and autocast for better performance on GPU.
    model = SpectraNet(X_train.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=6.58e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
    scaler = GradScaler()

    # Prepare dataset loaders
    train_loader = DataLoader(SpectraDataset(X_train, Y_train), batch_size=64, shuffle=True)
    val_loader = DataLoader(SpectraDataset(X_val, Y_val), batch_size=64)
    test_data = torch.tensor(X_test, dtype=torch.float32)
    test_targets = torch.tensor(Y_test, dtype=torch.float32)

    # Training the Model
    # Tracks validation vs training loss to determine if/when early stoppage criteria is met
    # Current patience is set to 10 epochs
    best_val_loss = float('inf')
    patience = 10
    counter = 0

    for epoch in range(150):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            with autocast(device_type=device.type):
                preds = model(x)
                loss = criterion(preds, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                val_loss += criterion(model(x), y).item() * x.size(0)

        avg_val_loss = val_loss / len(val_loader.dataset)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            counter = 0
            torch.save(model.state_dict(), f"..\\Saved-models\\best_model_{gas}_mc_dropout.pth")    # Saves most recent, best-performing model
        else:
            counter += 1
            if counter >= patience:
                print(f"Early Stopping Triggered at Epoch {epoch+1:02d}")
                break
        
    # Monte Carlo Dropout Evaluation
    # Evaluates CNN using Monte Carlo dropout with 600 forward passes
    # Final prediction is the mean of 600 predictions for each sample
    model.load_state_dict(torch.load(f"..\\Saved-models\\best_model_{gas}_mc_dropout.pth"))
    model.eval()

    def enable_dropout(m):
        if type(m) == nn.Dropout:
            m.train()

    model.apply(enable_dropout)

    # Normalize targets for figure
    # Data is normalized by subtracting the mean from the data and dividing by standard deviation
    mu = Y.mean()
    sigma = Y.std()
    test_targets_norm = ((test_targets - mu) / sigma).numpy()

    all_preds = []
    with torch.no_grad():
        for _ in range(600):
            preds = model(test_data.to(device)).cpu().numpy().flatten()
            preds_norm = (preds - mu) / sigma
            all_preds.append(preds_norm)

    # Evaluation metrics for model performance
    mc_mean = np.mean(all_preds, axis=0)
    mse = mean_squared_error(test_targets_norm, mc_mean)
    r2 = 1 - (np.sum((test_targets_norm - mc_mean) ** 2) / np.sum((test_targets_norm - np.mean(test_targets_norm)) ** 2))
    results.append({"gas": gas, "mse": mse, "r2": r2})

    # Plot of predicitions vs actuals for all test samples 
    plt.figure(figsize=(4, 3))
    plt.scatter(test_targets_norm, mc_mean, s=10, alpha=0.6)
    plt.plot([-2, 2], [-2, 2], 'r-', linewidth=1)

    plt.xlim(-1.5, 2)
    plt.ylim(-1.5, 2)
    plt.xlabel("True")
    plt.ylabel("Predictions")
    plt.title(f"{gas}")

    plt.xticks(fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout(pad=0.5)
    plt.savefig(f"..\\gas_plots\\{gas}_scatter.png")
    plt.close()

    end = datetime.now()
    elapsed = end - start
    print(f"{gas} Complete in {elapsed}")

    # Uncomment this line if wanting metrics printed at run-time
    #print(f"MSE: {mse:.5f} | R^2: {r2:.5f}")


=== Running Model for H2O ===
Early Stopping Triggered at Epoch 137
H2O Complete in 1:18:46.457601
MSE: 0.05747 | R^2: 0.94728

=== Running Model for CO2 ===
CO2 Complete in 1:27:33.838644
MSE: 0.09820 | R^2: 0.90430

=== Running Model for O2 ===
O2 Complete in 1:19:59.152530
MSE: 0.06518 | R^2: 0.93925

=== Running Model for N2 ===
Early Stopping Triggered at Epoch 58
N2 Complete in 0:32:54.375853
MSE: 0.87541 | R^2: 0.10490

=== Running Model for CH4 ===
Early Stopping Triggered at Epoch 92
CH4 Complete in 0:50:17.931400
MSE: 0.05236 | R^2: 0.94676

=== Running Model for N2O ===
Early Stopping Triggered at Epoch 126
N2O Complete in 1:07:24.746700
MSE: 0.10917 | R^2: 0.88628

=== Running Model for CO ===
Early Stopping Triggered at Epoch 101
CO Complete in 0:55:19.562422
MSE: 0.48703 | R^2: 0.48656

=== Running Model for O3 ===
O3 Complete in 1:19:42.363530
MSE: 0.03295 | R^2: 0.96737

=== Running Model for SO2 ===
SO2 Complete in 1:20:52.264446
MSE: 0.05521 | R^2: 0.94769

=== Runni

In [ ]:
results_df = pd.DataFrame(results)
results_df.to_csv("..\\CNN-evaluation-metrics\\gas_model_metrics.csv", index=False)